In [9]:
import cv2
from ultralytics import YOLO
import matplotlib.pyplot as plt
from inference import get_model
import supervision as sv
from inference_sdk import InferenceHTTPClient
from inference_sdk.webrtc import WebcamSource, StreamConfig, VideoMetadata


In [5]:
deed_model = YOLO("yolo11s.pt")  # โมเดลตั้งต้น
Result_Final_model = deed_model.train(
    data=r"C:\Users\user\Desktop\work\year 4\senior\110img_yolov11\data.yaml",
    epochs=300,
    imgsz=640,
    batch=16,
    device=0,
    verbose = True
)


New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.8  Python-3.9.23 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\user\Desktop\work\year 4\senior\110img_yolov11\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=

In [7]:
yolo_model = YOLO(r"C:\Users\user\Desktop\work\year 4\senior\runs\detect\train2\weights\best.pt")
img_file = r"C:\Users\user\Desktop\work\year 4\senior\110img_yolov11\test\images\DJI_20260120125325_0037_D_MP4-0302_jpg.rf.246521ca326e2c0c74b676aa83f71b32.jpg"
results = yolo_model.predict(source=img_file, save=True, conf=0.25)  # predict on an image file


image 1/1 C:\Users\user\Desktop\work\year 4\senior\110img_yolov11\test\images\DJI_20260120125325_0037_D_MP4-0302_jpg.rf.246521ca326e2c0c74b676aa83f71b32.jpg: 640x640 3 Buss, 13 carss, 7 motorcycles, 4 trucks, 1 tuktuk, 129.0ms
Speed: 5.3ms preprocess, 129.0ms inference, 5.8ms postprocess per image at shape (1, 3, 640, 640)
Results saved to C:\Users\user\Desktop\work\year 4\senior\runs\detect\predict


In [9]:
results

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Bus', 1: 'cars', 2: 'motorcycle', 3: 'truck', 4: 'tuktuk'}
 obb: None
 orig_img: array([[[146, 134, 128],
         [143, 131, 125],
         [143, 131, 125],
         ...,
         [112, 107, 108],
         [ 71,  65,  66],
         [186, 180, 181]],
 
        [[145, 133, 127],
         [143, 131, 125],
         [143, 131, 125],
         ...,
         [100,  95,  96],
         [133, 127, 128],
         [182, 176, 177]],
 
        [[145, 133, 127],
         [143, 131, 125],
         [145, 133, 127],
         ...,
         [ 88,  83,  84],
         [183, 178, 179],
         [144, 139, 140]],
 
        ...,
 
        [[170, 172, 173],
         [103, 105, 106],
         [ 62,  64,  65],
         ...,
         [136, 134, 133],
         [140, 138, 137],
         [144, 142, 141]],
 
        [[116, 118, 119],
         [ 68,  70,  71],
        

In [ ]:
# detect pics ver1
import cv2
import matplotlib.pyplot as plt

# Get the first result from results list
result = results[0]

# Get the original image
img = result.orig_img.copy()

# กำหนดรายชื่อคลาสที่ต้องการแสดงผล
target_classes = ["Bus", "cars", "mortorcycle", "truck", "tuktuk"]

# Draw bounding boxes forทุก class ที่กำหนด
for box in result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].int().tolist()
    conf = box.conf[0].item()
    cls = int(box.cls[0].item())
    class_name = result.names[cls]
    
    # Draw only if class อยู่ใน target_classes
    if class_name in target_classes:
        # Draw rectangle
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Put label
        label = f"{class_name} {conf:.2f}"
        cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

# Display the image
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.tight_layout()
plt.show()

# หากยังไม่แสดงผล ให้ลองบันทึกไฟล์ภาพด้วย
cv2.imwrite("output.jpg", img)
print("บันทึกภาพเป็น output.jpg แล้ว ลองเปิดดูในโฟลเดอร์เดียวกับ notebook")

<Figure size 1200x800 with 1 Axes>

บันทึกภาพเป็น output.jpg แล้ว ลองเปิดดูในโฟลเดอร์เดียวกับ notebook


In [ ]:
#detect video real-time ver1
import cv2
from ultralytics import YOLO

# 1. โหลดโมเดลที่คุณเทรนเสร็จแล้ว
model = YOLO(r"C:\Users\user\Desktop\work\year 4\senior\runs\detect\train2\weights\best.pt")

# 2. เปิดไฟล์วิดีโอ (หรือใส่ 0 เพื่อใช้กล้อง WebCam)
video_path = r"C:\Users\user\Desktop\vid_for_train\DJI_20260120125325_0037_D.MP4" 
cap = cv2.VideoCapture(video_path)

confidence_threshold = 0.6  # แสดงเฉพาะ box ที่มีความมั่นใจ >= 60%

# สร้างหน้าต่างแบบ resizable
cv2.namedWindow("YOLOv11 Real-time Detection (all classes, conf>=0.6)", cv2.WINDOW_NORMAL)
cv2.setWindowProperty("YOLOv11 Real-time Detection (all classes, conf>=0.6)", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

while cap.isOpened():
    success, frame = cap.read()
    
    if success:
        # 3. รันการ Detect ในแต่ละเฟรม
        results = model(frame, stream=True)

        for r in results:
            # 4. วาด Box สำหรับทุกคลาสที่มี conf >= 0.6
            boxes = r.boxes
            names = r.names
            annotated_frame = frame.copy()
            for box in boxes:
                cls = int(box.cls[0].item())
                class_name = names[cls]
                conf = box.conf[0].item()
                if conf >= confidence_threshold:
                    x1, y1, x2, y2 = box.xyxy[0].int().tolist()
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label = f"{class_name} {conf:.2f}"
                    cv2.putText(annotated_frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            # 5. แสดงผลหน้าต่างวิดีโอแบบเต็มจอ
            cv2.imshow("YOLOv11 Real-time Detection (all classes, conf>=0.6)", annotated_frame)

        # กด 'q' เพื่อออกจากลูป
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        break

cap.release()
cv2.destroyAllWindows()



0: 384x640 3 Buss, 72 carss, 15 motorcycles, 5 trucks, 86.0ms
Speed: 21.1ms preprocess, 86.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 Buss, 55 carss, 6 motorcycles, 3 trucks, 17.7ms
Speed: 4.3ms preprocess, 17.7ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Buss, 54 carss, 7 motorcycles, 3 trucks, 18.7ms
Speed: 3.5ms preprocess, 18.7ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Buss, 56 carss, 8 motorcycles, 5 trucks, 17.7ms
Speed: 2.3ms preprocess, 17.7ms inference, 3.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Buss, 33 carss, 1 motorcycle, 3 trucks, 19.5ms
Speed: 3.2ms preprocess, 19.5ms inference, 3.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Buss, 41 carss, 2 motorcycles, 4 trucks, 17.7ms
Speed: 2.8ms preprocess, 17.7ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Buss, 67 carss, 15 motorcycles,

KeyboardInterrupt: 

In [ ]:
import cv2
import base64
import numpy as np
from inference_sdk import InferenceHTTPClient
from inference_sdk.webrtc import VideoFileSource, StreamConfig, VideoMetadata

# 1. Initialize client
client = InferenceHTTPClient.init(
    api_url="http://localhost:9001",
    api_key="TUp8PX10FTmZOtFLOtDR"
)

# 2. ปรับเป็น realtime_processing=True เพื่อให้แสดงผลลัพธ์ตามเวลาจริง
# แต่ถ้าต้องการให้ AI เก็บทุกเฟรมครบๆ แม้จะช้า ให้เปลี่ยนเป็น False เหมือนเดิมครับ
source = VideoFileSource(
    r"C:\Users\user\Desktop\vid_for_train\DJI_20260120125325_0037_D.MP4", 
    realtime_processing=True 
)

VIDEO_OUTPUT = "output_image"
config = StreamConfig(
    stream_output=[], 
    data_output=["output_image", "predictions"]
)

session = client.webrtc.stream(
    source=source,
    workflow="small-object-detection-sahi",
    workspace="seniorproject-ujdsg",
    image_input="image",
    config=config
)

frames = []

# --- ฟังก์ชันแสดงผล Real-time ---
@session.on_data()
def on_data(data: dict, metadata: VideoMetadata):
    if VIDEO_OUTPUT in data:
        # 1. ถอดรหัสภาพจาก Base64
        img_data = base64.b64decode(data[VIDEO_OUTPUT]["value"])
        img_array = np.frombuffer(img_data, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        
        # 2. แสดงผลหน้าต่าง Real-time
        # ปรับขนาดหน้าจอลงหน่อยเผื่อวิดีโอต้นฉบับใหญ่เกินไป (เช่น 1280x720)
        display_img = cv2.resize(img, (1280, 720))
        cv2.imshow("SAHI Real-time Detection", display_img)
        
        # 3. เก็บเฟรมไว้ทำวิดีโอตอนท้าย (ถ้าต้องการ)
        timestamp_ms = metadata.pts * metadata.time_base * 1000
        frames.append((timestamp_ms, metadata.frame_id, img))
        
        print(f"Processed & Displaying frame {metadata.frame_id}")

        # 4. ปุ่มกดหยุด (กด 'q' ที่หน้าต่างวิดีโอเพื่อปิด)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            session.stop()
    else:
        print(f"Frame {metadata.frame_id}: No image data received")

# เริ่มรัน
print("🚀 Starting Real-time Inference... Press 'q' on the video window to stop.")
session.run()

# --- ส่วนท้าย: บันทึกวิดีโอ (Stitch frames) ---
if len(frames) > 0:
    print("🎬 Saving output video...")
    frames.sort(key=lambda x: x[1])
    # คำนวณ FPS จากเวลาจริงที่ได้รับ
    fps = (len(frames) - 1) / ((frames[-1][0] - frames[0][0]) / 1000) if len(frames) > 1 else 30.0
    h, w = frames[0][2].shape[:2]
    out = cv2.VideoWriter("output_sahi.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for _, _, frame in frames:
        out.write(frame)
    out.release()
    print(f"✅ Done! saved to output_sahi.mp4")

cv2.destroyAllWindows()

🚀 Starting Real-time Inference... Press 'q' on the video window to stop.


INFO [inference_sdk.webrtc.session] ICE connection state: checking
INFO [inference_sdk.webrtc.session] Connection state: connecting
INFO [inference_sdk.webrtc.session] ICE connection state: completed
INFO [inference_sdk.webrtc.session] Connection state: connected


In [ ]:
#no couting line
import cv2
from ultralytics import YOLO

# 1. โหลดโมเดล
model_path = r"C:\Users\user\Desktop\work\year 4\senior\runs\detect\train2\weights\best.pt"
model = YOLO(model_path)

# 2. เปิดไฟล์วิดีโอ
video_path = r"C:\Users\user\Desktop\vid_for_train\DJI_20260120125325_0037_D.MP4"
cap = cv2.VideoCapture(video_path)

# ตั้งค่าสี (BGR)
color_map = {
    "Bus": (0, 0, 255),         # แดง
    "cars": (0, 255, 0),        # เขียว
    "motorcycle": (255, 0, 255), # ม่วง
    "tuktuk": (0, 165, 255),    # ส้ม
    "truck": (255, 0, 0)        # น้ำเงิน
}
default_color = (255, 255, 255)
conf_threshold = 0.5 

# ขนาดเฟรมที่เรากำหนดไว้
FRAME_W, FRAME_H = 1280, 720

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.resize(frame, (FRAME_W, FRAME_H))
    results = model.predict(frame, conf=conf_threshold, verbose=False)
    
    current_counts = {}
    
    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0].item())
            class_name = r.names[cls_id]
            conf = float(box.conf[0].item())
            box_color = color_map.get(class_name, default_color)
            
            current_counts[class_name] = current_counts.get(class_name, 0) + 1
            x1, y1, x2, y2 = box.xyxy[0].int().tolist()
            
            cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
            label = f"{class_name} {conf:.2f}"
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)

    # --- ส่วนที่แก้ไข: ย้ายตารางไปขอบล่างขวา ---
    if current_counts:
        # คำนวณขนาดตาราง
        rect_w = 300
        rect_h = 50 + (len(current_counts) * 35)
        
        # กำหนดพิกัดมุมซ้ายบนของตาราง (ให้ห่างจากขอบล่างและขวาอย่างละ 20 พิกเซล)
        bg_x1 = FRAME_W - rect_w - 20
        bg_y1 = FRAME_H - rect_h - 20
        bg_x2 = FRAME_W - 20
        bg_y2 = FRAME_H - 20

        # วาดพื้นหลังตาราง (สีดำ โปร่งแสงนิดๆ ด้วยการใช้ Rectangle)
        cv2.rectangle(frame, (bg_x1, bg_y1), (bg_x2, bg_y2), (0, 0, 0), -1)
        
        # เขียนหัวข้อ
        y_text = bg_y1 + 30
        cv2.putText(frame, "Vehicle Count:", (bg_x1 + 15, y_text), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        # เขียนรายการคลาส
        for cls_name, count in current_counts.items():
            y_text += 35
            text_color = color_map.get(cls_name, (255, 255, 255))
            text = f"{cls_name}: {count}"
            cv2.putText(frame, text, (bg_x1 + 25, y_text), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2)

    cv2.imshow("Local YOLOv11 Vehicle Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
#change cumulative counting ver1
import cv2
from ultralytics import YOLO

# 1. โหลดโมเดล
model = YOLO(r"C:\Users\user\Desktop\work\year 4\senior\runs\detect\train2\weights\best.pt")

# 2. เปิดไฟล์วิดีโอ
video_path = r"C:\Users\user\Desktop\vid_for_train\DJI_20260120125325_0037_D.MP4"
cap = cv2.VideoCapture(video_path)

# --- ตั้งค่าสำหรับการนับสะสม ---
line_y = 500  # ตำแหน่งความสูงของเส้นนับ (ปรับตามความเหมาะสมของวิดีโอ)
cumulative_counts = {}  # เก็บจำนวนสะสมแยกตามคลาส {"cars": 5, "Bus": 2}
tracked_ids = set()     # เก็บ ID ที่เคยนับไปแล้ว เพื่อไม่ให้นับซ้ำ

color_map = {
    "Bus": (0, 0, 255), "cars": (0, 255, 0), "motorcycle": (255, 0, 255),
    "tuktuk": (0, 165, 255), "truck": (255, 0, 0)
}

while cap.isOpened():
    success, frame = cap.read()
    if not success: break
    frame = cv2.resize(frame, (1280, 720))

    # 3. รัน Tracking ด้วย ByteTrack
    # persist=True คือการบอกให้จำ ID ต่อเนื่องจากเฟรมก่อนหน้า
    results = model.track(frame, persist=True, tracker="bytetrack.yaml", conf=0.5, verbose=False)

    # วาดเส้นสำหรับนับ
    cv2.line(frame, (0, line_y), (1280, line_y), (255, 255, 255), 2)
    cv2.putText(frame, "Counting Line", (10, line_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.int().cpu().tolist()
        class_ids = results[0].boxes.cls.int().cpu().tolist()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        names = results[0].names

        for box, cls_id, track_id in zip(boxes, class_ids, track_ids):
            class_name = names[cls_id]
            x1, y1, x2, y2 = box
            center_y = int((y1 + y2) / 2) # หาจุดกึ่งกลางของรถ
            
            # 4. Logic การนับสะสม: ถ้าจุดกึ่งกลางรถข้ามเส้น และยังไม่เคยนับ ID นี้มาก่อน
            if center_y > line_y and track_id not in tracked_ids:
                cumulative_counts[class_name] = cumulative_counts.get(class_name, 0) + 1
                tracked_ids.add(track_id) # บันทึกว่านับ ID นี้ไปแล้ว

            # วาดกรอบและ ID
            color = color_map.get(class_name, (255, 255, 255))
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID:{track_id} {class_name}", (x1, y1 - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # 5. แสดงตารางนับสะสมที่มุมล่างขวา
    rect_h = 50 + (len(cumulative_counts) * 35)
    bg_x1, bg_y1 = 960, 720 - rect_h - 20
    cv2.rectangle(frame, (bg_x1, bg_y1), (1260, 700), (0, 0, 0), -1)
    
    curr_y = bg_y1 + 30
    cv2.putText(frame, "CUMULATIVE TOTAL:", (bg_x1 + 10, curr_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    for cls_name, count in cumulative_counts.items():
        curr_y += 35
        cv2.putText(frame, f"{cls_name}: {count}", (bg_x1 + 20, curr_y), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_map.get(cls_name, (255, 255, 255)), 2)

    cv2.imshow("ByteTrack Cumulative Counting", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [18]:
#diagonal line counting ver1
import cv2
import numpy as np
from ultralytics import YOLO

# 1. โหลดโมเดล
model = YOLO(r"C:\Users\user\Desktop\work\year 4\senior\runs\detect\train2\weights\best.pt")

# 2. เปิดไฟล์วิดีโอ
cap = cv2.VideoCapture(r"C:\Users\user\Desktop\vid_for_train\DJI_20260120125325_0037_D.MP4")

# --- ตั้งค่าจุดสำหรับเส้นเฉียง ---
line_points = [] # เก็บพิกัด [(x1, y1), (x2, y2)]
cumulative_counts = {}
tracked_ids = set()

def draw_line_event(event, x, y, flags, param):
    global line_points
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(line_points) >= 2: # ถ้ามีครบ 2 จุดแล้วให้เริ่มใหม่
            line_points = []
        line_points.append((x, y))
        print(f"📍 กำหนดจุดที่ {len(line_points)}: ({x}, {y})")

cv2.namedWindow("Diagonal Line Counting")
cv2.setMouseCallback("Diagonal Line Counting", draw_line_event)

# ฟังก์ชันตรวจสอบว่าจุด (px, py) อยู่ "เหนือ" หรือ "ใต้" เส้นเฉียง
def is_below_line(px, py, x1, y1, x2, y2):
    # ใช้ Cross Product: (x2-x1)*(py-y1) - (y2-y1)*(px-x1)
    # ผลลัพธ์จะเป็นบวกหรือลบ ขึ้นอยู่กับตำแหน่งของจุดเทียบกับเส้น
    return (x2 - x1) * (py - y1) - (y2 - y1) * (px - x1) > 0

color_map = {"Bus": (0,0,255), "cars": (0,255,0), "motorcycle": (255,0,255), "tuktuk": (0,165,255), "truck": (255,0,0)}

while cap.isOpened():
    success, frame = cap.read()
    if not success: break
    frame = cv2.resize(frame, (1280, 720))

    # วาดเส้นเฉียง
    if len(line_points) == 2:
        cv2.line(frame, line_points[0], line_points[1], (255, 255, 255), 2)
        cv2.putText(frame, "Diagonal Counting Line", line_points[0], cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    results = model.track(frame, persist=True, tracker="bytetrack.yaml", conf=0.5, verbose=False)

    if results[0].boxes.id is not None and len(line_points) == 2:
        boxes = results[0].boxes.xyxy.int().cpu().tolist()
        class_ids = results[0].boxes.cls.int().cpu().tolist()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        
        lx1, ly1 = line_points[0]
        lx2, ly2 = line_points[1]

        for box, cls_id, track_id in zip(boxes, class_ids, track_ids):
            class_name = results[0].names[cls_id]
            x1, y1, x2, y2 = box
            cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2) # จุดกึ่งกลางรถ
            
            # ตรวจสอบการข้ามเส้นแนวเฉียง
            # หากรถ "ข้ามเส้น" (เปลี่ยนฝั่งจาก False เป็น True) ให้ทำการนับ
            if is_below_line(cx, cy, lx1, ly1, lx2, ly2) and track_id not in tracked_ids:
                cumulative_counts[class_name] = cumulative_counts.get(class_name, 0) + 1
                tracked_ids.add(track_id)

            color = color_map.get(class_name, (255, 255, 255))
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID:{track_id} {class_name}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # แสดงตาราง (มุมล่างขวา)
    rect_h = 50 + (len(cumulative_counts) * 35)
    cv2.rectangle(frame, (960, 720 - rect_h - 20), (1260, 700), (0, 0, 0), -1)
    curr_y = 720 - rect_h + 10
    for cls_name, count in cumulative_counts.items():
        cv2.putText(frame, f"{cls_name}: {count}", (980, curr_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_map.get(cls_name, (255,255,255)), 2)
        curr_y += 35

    cv2.imshow("Diagonal Line Counting", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

📍 กำหนดจุดที่ 1: (652, 471)
📍 กำหนดจุดที่ 2: (790, 563)
📍 กำหนดจุดที่ 1: (506, 563)
📍 กำหนดจุดที่ 2: (625, 475)
📍 กำหนดจุดที่ 1: (539, 112)
📍 กำหนดจุดที่ 2: (752, 120)
📍 กำหนดจุดที่ 1: (174, 410)
📍 กำหนดจุดที่ 2: (652, 32)
